In [25]:
import torch
from torch import Tensor
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pickle  # TODO implement pickalable training loop
import pandas as pd
import random
import os
import copy

# static typing - vscode has good built in extensions for it
from numpy.typing import NDArray

from typing import Literal, Generator, Callable, TypeAlias

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Load the data

Data is divided into groups of tasks.  
For each group, we look for data that corresponds to this group.  
Then we divide the data into multiple dataloaders ; 1 per group.

### Hyperparameters

In [26]:
data_path = "./scratch/Malware/CICAndMal/processed_data/"  # folder containing the npy files
divided_into_tasks_data_path: str = os.path.join(data_path, "divided_by_tasks")


nb_cl_first_group = 22  # number of classes in the first training, mentioned in 5.2. Should be tested
nb_groups = 5  # nb of groups of classes
nb_cl_per_group = 5  # nb of classes per group (except first group)
total_classes = 42  # number of labels for our NN

in_features = 85  # X_data has 85 features for CICAndMal

In [27]:
def pre_divide_data_by_task(total_classes: int, data_path: str, divided_into_tasks_data_path: str, tag: Literal["train", "test", "valid"]):
    """Step in data processing where we already divide it into tasks to make fetching the data faster and easier.

    Args:
        total_classes (int): number of classes in the data
        data_path (str): folder leading to .npy files
        tag: train test or valid
    """

    if not os.path.exists(divided_into_tasks_data_path):
        os.makedirs(divided_into_tasks_data_path)

    # data path
    X_path: str = f"{data_path}/X_{tag}.npy"
    y_path: str = f"{data_path}/y_{tag}.npy"
    X_data: NDArray[np.float64] = np.load(X_path)
    y_data: NDArray[np.int64] = np.load(y_path)

    print(f"{y_data.shape} rows")

    for i in range(total_classes):
        task_indices = np.where(y_data == i)[0]  # Every index i for which y_data[i] is in the array g

        X_i: NDArray[np.float64] = X_data[task_indices]

        # sanity check
        # y_i: NDArray[np.int64] = y_data[task_indices]
        # assert (y_i == y_i[0]).all()

        np.save(f"{divided_into_tasks_data_path}/X_{i}_{tag}.npy", X_i)


pre_divide_data_by_task(total_classes, data_path, divided_into_tasks_data_path, "train")
pre_divide_data_by_task(total_classes, data_path, divided_into_tasks_data_path, "test")

(1105290,) rows
(138162,) rows


In [28]:
def load_task_i(divided_into_tasks_data_path: str, task: int, tag: Literal["train", "test", "valid"]) -> NDArray[np.float64]:
    X: NDArray[np.float64] = np.load(f"{divided_into_tasks_data_path}/X_{task}_{tag}.npy")
    return X

In [29]:
def divide_classes_into_groups(total_classes: int, nb_cl_first_group: int, nb_groups: int, nb_cl_per_group: int) -> list[NDArray[np.int64]]:
    """return the classes divided into random groups of predermined size

    Args:
        total_classes (int): number of classes
        nb_cl_first_group (int): number of classes in first group
        nb_groups (int): nb of groups including the first one
        nb_cl_per_group (int): number of classes per group except the first one

    Returns:
        list[NDArray[np.int64]]: _description_
    """
    assert total_classes == nb_cl_first_group + (
        (nb_groups - 1) * nb_cl_per_group
    ), "verify we have the correct number of classes in the group division"

    classes_order: NDArray[np.int64] = np.arange(total_classes)
    np.random.shuffle(classes_order)

    groups: list[NDArray[np.int64]] = []
    groups.append(classes_order[:nb_cl_first_group])
    for i in range(nb_groups - 1):
        group_order_index_start: int = nb_cl_first_group + i * nb_cl_per_group
        groups.append(classes_order[group_order_index_start : group_order_index_start + nb_cl_per_group])
    return groups


print(divide_classes_into_groups(total_classes, nb_cl_first_group, nb_groups, nb_cl_per_group))

[array([ 3, 11, 31, 13, 33,  4, 29, 12, 34, 20,  1, 19, 26, 15, 25,  8, 39,
       40,  0, 37,  6, 28]), array([ 2, 35,  5, 22, 18]), array([38, 17, 32, 10, 41]), array([14, 16,  7, 30, 21]), array([27, 36, 24,  9, 23])]


In [30]:
class GroupTrainingSet:
    """Just making this class to add clarity with the names from the paper
    Corresponds to D_t from the algorithm on page 4.
    """

    def __init__(self, X_g_list: list[NDArray[np.float64]], y_g_list: list[int]) -> None:
        self.X_g_list: list[NDArray[np.float64]] = X_g_list
        self.y_g_list: list[int] = y_g_list

    def __len__(self):
        return len(self.X_g_list)
    
    def nb_entries(self):
        return sum([X.shape[0] for X in self.X_g_list])

    def get_tensors(self) -> tuple[NDArray[np.float64], NDArray[np.int64]]:
        X_combined: NDArray[np.float64] = np.concatenate(self.X_g_list, axis=0)
        y_combined: NDArray[np.int64] = np.concatenate(
            [np.full(X.shape[0], task, dtype=np.int64) for X, task in zip(self.X_g_list, self.y_g_list)], axis=0
        )
        return X_combined, y_combined


def load_groups_data(
    divided_into_tasks_data_path: str, groups: list[NDArray[np.int64]], tag: Literal["train", "test", "valid"]
) -> Generator[GroupTrainingSet, None, None]:
    """For each group, load and yield the corresponding data from pre-divided task files

    Args:
        divided_into_tasks_data_path (str): path to predivided data
        groups (list[NDArray[np.int64]]): groups
        tag: train test or valid

    Yields:
        Generator[GroupTrainingSet], None, None]: For each group, yield a GroupTrainingSet containing the data from every task of the group
    """

    for g in groups:
        X_g_list: list[NDArray[np.float64]] = [load_task_i(divided_into_tasks_data_path, task, tag) for task in g]
        yield GroupTrainingSet(X_g_list, g.tolist())


def tests_dataloader(divided_into_tasks_data_path, tasks: list[int])-> DataLoader:
    """Dataloader for the tasks already trained on

    Args:
        divided_into_tasks_data_path (_type_): path to divided data
        tasks (list[int]): list of trained tasks

    Returns:
        DataLoader: DataLoader
    """
    X_list = [load_task_i(divided_into_tasks_data_path, task, "test") for task in tasks]
    X_combined: NDArray[np.float64] = np.concatenate(X_list, axis=0)
    y_combined: NDArray[np.int64] = np.concatenate([np.full(X.shape[0], task, dtype=np.int64) for X, task in zip(X_list, tasks)], axis=0)
    dataset = TensorDataset(torch.from_numpy(X_combined).float(), torch.from_numpy(y_combined).long())
    return DataLoader(dataset, batch_size=32, shuffle=False)


# NN Models

For now I just copied their model from their paper.

In [31]:
import torch
import torch.nn as nn


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, feats: int, head: int = 8, dropout: float = 0.0):
        super().__init__()
        self.head = head
        self.feats = feats
        # self.sqrt_d = feats ** 0.5
        self.sqrt_d = (feats // head) ** 0.5

        self.q = nn.Linear(feats, feats)
        self.k = nn.Linear(feats, feats)
        self.v = nn.Linear(feats, feats)

        self.o = nn.Linear(feats, feats)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, n, f = x.size()
        h = self.head
        d = f // h

        # project & split heads
        q = self.q(x).view(b, n, h, d).transpose(1, 2)
        k = self.k(x).view(b, n, h, d).transpose(1, 2)
        v = self.v(x).view(b, n, h, d).transpose(1, 2)

        # scaled dot-product
        attn = torch.softmax((q @ k.transpose(-2, -1)) / self.sqrt_d, dim=-1)  # (b,h,n,n)

        # combine
        out = (attn @ v).transpose(1, 2).contiguous().view(b, n, f)  # (b,n,f)
        return self.o(self.dropout(out))


class TransformerEncoder(nn.Module):
    def __init__(self, feats: int, mlp_hidden: int, head: int = 8, dropout: float = 0.0):
        super().__init__()
        self.la1 = nn.LayerNorm(feats)
        self.msa = MultiHeadSelfAttention(feats, head=head, dropout=dropout)
        self.la2 = nn.LayerNorm(feats)
        self.mlp = nn.Sequential(
            nn.Linear(feats, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, feats),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        out = self.msa(self.la1(x)) + x
        out = self.mlp(self.la2(out)) + out
        return out


class EmberTransformer(nn.Module):
    def __init__(
        self,
        in_feats: int = 85,
        num_classes: int = 42,
        hidden: int = 384,
        mlp_hidden: int = 384 * 3,
        num_layers: int = 6,
        nhead: int = 8,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.feat_emb = nn.Linear(in_feats, hidden)
        self.use_cls = True
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden))
        n_tokens = 2
        self.pos_emb = nn.Parameter(torch.randn(1, n_tokens, hidden))

        # Stack of TransformerEncoders
        encoders = [TransformerEncoder(hidden, mlp_hidden, head=nhead, dropout=dropout) for _ in range(num_layers)]
        self.encoder = nn.Sequential(*encoders)

        # Final classifier
        self.classifier = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, num_classes))

    def forward(self, x):
        """
        x: (B, D) Ember features
        returns logits (B, C), and optionally features for contrastive/loss (B, hidden)
        """
        B = x.size(0)
        # (B, hidden) -> (B, 1, hidden)
        tokens = self.feat_emb(x).unsqueeze(1)

        if self.use_cls:
            # (B, 1, hidden)
            cls = self.cls_token.expand(B, -1, -1)
            tokens = torch.cat([cls, tokens], dim=1)  # now (B, 2, hidden)

        # add pos embed
        tokens = tokens + self.pos_emb

        # run through transformer stack
        out = self.encoder(tokens)  # (B, n_tokens, hidden)

        if self.use_cls:
            feat = out[:, 0]  # (B, hidden)
        else:
            feat = out[:, 0]  # same shape

        logits = self.classifier(feat)  # (B, num_classes)
        return logits

    def feature_extract(self, x):
        """
        Extract features without classification
        """
        B = x.size(0)
        # (B, hidden) -> (B, 1, hidden)
        tokens = self.feat_emb(x).unsqueeze(1)

        if self.use_cls:
            # (B, 1, hidden)
            cls = self.cls_token.expand(B, -1, -1)
            tokens = torch.cat([cls, tokens], dim=1)  # now (B, 2, hidden)

        # add pos embed
        tokens = tokens + self.pos_emb

        # run through transformer stack
        out = self.encoder(tokens)  # (B, n_tokens, hidden)

        if self.use_cls:
            feat = out[:, 0]  # (B, hidden)
        else:
            feat = out[:, 0]  # same shape

        return feat
    
    def __str__(self):
        return "EmberTransformer"

In [32]:
class SimpleMLP(nn.Module):
    def __init__(self, in_features: int = 85, num_classes: int = 42, hidden_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )
    
    def forward(self, x):
        logits = self.mlp(x)
        return logits
    
    def __str__(self):
        return "SimpleMLP"

In [33]:
class SimplestMLP(nn.Module):
    def __init__(self, in_features: int = 85, num_classes: int = 42, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits
    
    def __str__(self):
        return "SimplestMLP"

In [34]:
# Section 4.2

ExamplarSelectionStrategy: TypeAlias = Callable[[NDArray[np.float64], int], NDArray[np.float64]]


class Buffer:
    def __init__(
        self,
        buffer_truncate_class: ExamplarSelectionStrategy,
        new_task_select_m_exemplars: ExamplarSelectionStrategy,
        K: int,
    ) -> None:
        # TODO initialize non-empty buffer with pickle
        """Initialize empty buffer

        Args:
            buffer_truncate_class (ExamplarSelectionStrategy): Function to update each element of the buffer
            buffer_update (ExamplarSelectionStrategy): Function to decide what m elements to add in the buffer for each class, strategies in 4.1
            K: buffer capacity in 4.2
        """
        self.X_tasks_list: list[NDArray[np.float64]] = []
        self.y_tasks_list: list[int] = []
        # TODO initialize non-empty buffer with pickle

        self.K: int = K
        self.buffer_truncate_class: ExamplarSelectionStrategy = buffer_truncate_class
        self.new_task_select_m_exemplars: ExamplarSelectionStrategy = new_task_select_m_exemplars

        self.last_added_tasks: list[int] = []  # Keep track of the last group of tasks added to the buffer for refinement

    def get_tensors(self):
        if not self.X_tasks_list:
            return None, None

        X_combined: NDArray[np.float64] = np.concatenate(self.X_tasks_list, axis=0)
        y_combined: NDArray[np.int64] = np.concatenate(
            [np.full(X.shape[0], task, dtype=np.int64) for X, task in zip(self.X_tasks_list, self.y_tasks_list)], axis=0
        )

        return X_combined, y_combined

    def update_buffer(self, D_t: GroupTrainingSet):
        nb_classes_in_buffer: int = len(self.X_tasks_list)
        nb_classes_in_new_dataset = len(D_t)

        nb_classes = nb_classes_in_buffer + nb_classes_in_new_dataset
        m: int = self.K // nb_classes

        self.X_tasks_list = [self.buffer_truncate_class(class_in_buffer, m) for class_in_buffer in self.X_tasks_list]
        new_X_tasks = [self.new_task_select_m_exemplars(new_task_in_buffer, m) for new_task_in_buffer in D_t.X_g_list]
        self.X_tasks_list.extend(new_X_tasks)

        self.y_tasks_list.extend(D_t.y_g_list)
        self.last_added_tasks = D_t.y_g_list

    def dataloader(self, batch_size=32, shuffle=True) -> DataLoader:
        X, y = self.get_tensors()
        dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [35]:
def get_dataloader_from_data_and_buffer(D_t: GroupTrainingSet, E: Buffer, batch_size: int = 32, shuffle: bool = True) -> DataLoader:
    """Returns a dataloader with the buffer included as specified in phase 1 initial training of the paper

    Args:
        D_t (GroupTrainingSet): Data for the training of the group
        E (Buffer): Data from the buffer
        batch_size (int, optional): batch size. Defaults to 32.
        shuffle (bool, optional): shuffle the data. Defaults to True.

    Returns:
        DataLoader: dataloader of the tensors.
    """
    X_buffer, y_buffer = E.get_tensors()
    X_D_t, y_D_t = D_t.get_tensors()
    if X_buffer is None or y_buffer is None:  # Edge case when buffer still empty
        X = X_D_t
        y = y_D_t
    else:
        X: NDArray[np.float64] = np.concatenate([X_D_t, X_buffer], axis=0)
        y: NDArray[np.int64] = np.concatenate([y_D_t, y_buffer], axis=0)

    dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [40]:
# Section 4.*
from torch._tensor import Tensor
from torch.nn.modules.module import Module
from torch.optim.optimizer import Optimizer


class TraMEL:
    def __init__(self, model: Module, E: Buffer, optimizer: torch.optim.Optimizer) -> None:
        """TraMEL algorithm keeps track of model and a buffer E that updates over time"""
        self.model: Module = model.to(device)  # We will try different models, relevant to section 4.4
        self.f_i_m1: Module = copy.deepcopy(self.model).to(device)  # refinement section 4.3
        self.f_prime: Module = copy.deepcopy(self.model).to(device)  # refinement section 4.3
        self.E: Buffer = E
        self.optimizer: Optimizer = optimizer
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        
        
        self.group = 0
        self.epoch = 0
        self.phase: Literal[1,2,3,4] = 1


    def range(self, epochs: int):
        """Custom range method, just to keep the current epoch in pickle and keep the *phases* methods clean with no pickle management"""
        for i in range(self.epoch, epochs):
            self.epoch = i
            self.dump()
            yield i
        self.epoch = 0

    def phase1_train(
        self, D_t: GroupTrainingSet, *, epochs: int = 50
    ):  # LIST AS PLACEHOLDER, TODO find correct data type D_t represents the t-th dataset
        """Train model on D_t and E"""
        self.model.train()
        dataloader: DataLoader = get_dataloader_from_data_and_buffer(D_t, self.E)
        for epoch in self.range(epochs):
            epoch_losses = []

            for X, y in dataloader:
                X: Tensor = X.to(device)
                y: Tensor = y.to(device)

                self.optimizer.zero_grad()
                logits = self.model(X)
                loss = self.loss_fn(logits, y)
                loss.backward()
                self.optimizer.step()

                epoch_losses.append(loss.item())

            print(epoch, "epoch", "batch", f"Average loss: {np.mean(epoch_losses[-100:]):.6f}")

    def phase2_examplar_selection(self, D_t: GroupTrainingSet):
        self.E.update_buffer(D_t)

    def phase3_refinement(self, t: int, *,alpha: int = 1, beta: int = 1, epochs=20):
        """Refine the model with the buffer which contains examples from the current dataset and the previous ones.

        Args:
            t (int): timestep/index of current group
            alpha (int, optional): Section 4.3. Defaults to 1.
            beta (int, optional): Section 4.3. Defaults to 1.
        """
        if t == 0:
            self.f_i_m1.load_state_dict(self.model.state_dict())
            return

        self.f_prime.load_state_dict(self.model.state_dict())
        dataloader: DataLoader = self.E.dataloader()

        mse = nn.MSELoss(reduction="mean")
        ce = nn.CrossEntropyLoss(reduction="mean")

        last_added_tensor: Tensor = torch.tensor(self.E.last_added_tasks, dtype=torch.long).to(device)

        self.model.train()
        self.f_i_m1.eval()
        self.f_prime.eval()

        for epoch in self.range(epochs):
            epoch_losses = []

            for X, y in dataloader:
                X: Tensor = X.to(device)
                y: Tensor = y.to(device)

                E_i_mask: Tensor = torch.isin(y, last_added_tensor)  # get a boolean tensor telling us which tasks were just added to the buffer
                E_bi_mask: Tensor = (
                    ~E_i_mask
                )  # get a boolean tensor telling us which tasks were in the buffer before, (on which self.f_i_m1 is trained)

                fix: Tensor = self.model(X)  # f^{i}(x)
                with torch.no_grad():
                    fim1x: Tensor = self.f_i_m1(X)  # f^{i-1}(x)
                    fip: Tensor = self.f_prime(X)  # f^{i}'(x)

                l_ce: Tensor = ce(fix, y)
                # if there are entries corresponding to the old tasks in this batch
                l_past: Tensor = mse(fix[E_bi_mask], fim1x[E_bi_mask]) if E_bi_mask.any() else torch.tensor(0.0).to(device)
                l_current: Tensor = mse(fix[E_i_mask], fip[E_i_mask]) if E_i_mask.any() else torch.tensor(0.0).to(device)

                loss: Tensor = l_ce + alpha * l_past + beta * l_current

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                epoch_losses.append(loss.item())

            print(epoch, "epoch", "batch", f"Average loss: {np.mean(epoch_losses[-100:]):.6f}")
        self.f_i_m1.load_state_dict(self.model.state_dict())

    def phase4_test(self, divided_into_tasks_data_path: str) -> float:
        """Test the model on all seen tasks and compute overall accuracy
        
        Args:
            divided_into_tasks_data_path: Path to divided task data
            
        Returns:
            float: Overall accuracy on all seen tasks
        """
        self.model.eval()
        seen_tasks = self.E.y_tasks_list
        test_dataloader = tests_dataloader(divided_into_tasks_data_path, seen_tasks)
        
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for X, y in test_dataloader:
                X = X.to(device)
                y = y.to(device)
                
                logits = self.model(X)
                preds = torch.argmax(logits, dim=1)
                
                all_preds.append(preds.cpu().numpy())
                all_labels.append(y.cpu().numpy())
        
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        
        accuracy = np.mean(all_preds == all_labels)
        print(f"Test Accuracy: {accuracy:.4f}")
        
        return accuracy
    

    ################ Pickle ####################
    def dump(self, name: str = "tramel.pickle") -> None:
        print("dumped")
        with open(name, 'wb') as f:
                pickle.dump(self, f)

    @classmethod
    def load(cls, name: str = "tramel.pickle") -> "TraMEL":
        with open(name, 'rb') as f:
            return pickle.load(f)
        
    def dump_first_group(self):
        """Since first group specifically takes a long time to train
            and does not test what the paper is about (catastrophic forgetting). We dont train it too much"""
        self.dump("first_group.pickle")

    @classmethod
    def load_first_group(cls) -> "TraMEL":
         return cls.load("first_group.pickle")
    
    def __str__(self):
        return f"""group: {self.group}
epoch: {self.epoch}
phase: {self.phase}
model: {self.model}
"""


In [37]:
def random_exemplar_selection(X_task: NDArray[np.float64], m: int) -> NDArray[np.float64]:
    """Select m random exemplars from X_task"""
    if m > X_task.shape[0]:
        return X_task
    n_samples: int = X_task.shape[0]
    indices: NDArray[np.int64] = np.random.choice(n_samples, size=m, replace=False)
    ret = X_task[indices]
    assert len(ret) == m
    return ret

# Training loop

In [ ]:
try:
    tramel : TraMEL = TraMEL.load()
except FileNotFoundError:
    try:
        tramel : TraMEL = TraMEL.load_first_group()
        print("Pickle not found, starting from first group")
    except FileNotFoundError:
        print("No pickle found starting from fresh instance")
        
        model = EmberTransformer()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.00001)
        K = 30000
        E = Buffer(random_exemplar_selection, random_exemplar_selection, K)
        tramel = TraMEL(model, E, optimizer)

print(tramel)
    


groups: list[NDArray[np.int64]] = divide_classes_into_groups(total_classes, nb_cl_first_group, nb_groups, nb_cl_per_group)
for t, D_t in enumerate(load_groups_data(divided_into_tasks_data_path, groups, "train")):
    if t < tramel.group:
        # skip the first few groups if they are already trained in the tramel
        continue
    else:
        tramel.group = t

    print(f"nb of entries in group: {D_t.nb_entries()}")
    if tramel.phase == 1:
        tramel.phase1_train(D_t, epochs=30)
        tramel.phase = 2
        tramel.dump()
    
    if tramel.phase == 2:
        tramel.phase2_examplar_selection(D_t)
        tramel.phase = 3
        tramel.dump()

    if tramel.phase == 3:
        tramel.phase3_refinement(t, epochs=10)
        tramel.phase = 4
        tramel.dump()

    if tramel.phase == 4:
        tramel.phase4_test(divided_into_tasks_data_path)
        tramel.phase = 1
        tramel.dump()

    if t == 0:
        tramel.dump_first_group()

group: 0
epoch: 5
phase: 1
model: EmberTransformer

nb of entries in group: 494122
dumped
5 epoch batch Average loss: 2.013608
dumped
6 epoch batch Average loss: 1.983774
dumped
7 epoch batch Average loss: 1.942601
dumped
8 epoch batch Average loss: 1.821016
dumped
9 epoch batch Average loss: 1.810059
dumped
10 epoch batch Average loss: 1.778902
dumped
11 epoch batch Average loss: 1.741955
dumped
12 epoch batch Average loss: 1.698129
dumped
13 epoch batch Average loss: 1.698763
dumped
14 epoch batch Average loss: 1.651740
dumped
15 epoch batch Average loss: 1.608637
dumped
16 epoch batch Average loss: 1.643057
dumped
17 epoch batch Average loss: 1.561795
dumped


KeyboardInterrupt: 